# [Data Rescue Project datasets](https://portal.datarescueproject.org/datasets/)

Check the statuses of the data sources.


## Setup

Adjust import path:


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import httpx

from ptps_wildfire_demo import Resolver

client = httpx.AsyncClient()
resolver = Resolver(client)

## Datasets

Get a sample:


In [3]:
# make this repeatable
SEED = 1

sample = resolver.drp_rescues.copy()

# some URLs are missing the protocol
is_http = sample["data_source"].str.startswith("http://")
is_https = sample["data_source"].str.startswith("https://")
sample.loc[~(is_http | is_https), "data_source"] = "https://" + sample["data_source"]

# get rid of duplicate URLs
sample = sample.drop_duplicates("data_source")
sample = sample.sample(300, random_state=SEED)
sample = sample.sort_values("data_source")

sample

,title,organization,agency,description,data_source,dataset_source_status,websites,metadata_available,metadata_url,category,last_modified,url,resources
28,2019 Farm to School Census v2,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251022062649/http...,"[Agriculture, Science & Research]",2026-07-29,/datasets/2019-farm-to-school-census-v2/,"[{'id': 3848, 'status': 'Finished', 'download_..."
735,CSR2 Study for Greenhouse gas Reduction throug...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20250911060110/http...,"[Agriculture, Science & Research]",2026-07-20,/datasets/csr2-study-for-greenhouse-gas-reduct...,"[{'id': 3342, 'status': 'Finished', 'download_..."
527,Chelonus insularis Official Gene Set OGSv1.0,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251205021337/http...,"[Agriculture, Science & Research]",2026-07-27,/datasets/chelonus-insularis-official-gene-set...,"[{'id': 3626, 'status': 'Finished', 'download_..."
647,Comparison of four extractants used in soil ph...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251116210425/http...,"[Agriculture, Science & Research]",2026-07-20,/datasets/comparison-of-four-extractants-used-...,"[{'id': 3371, 'status': 'Finished', 'download_..."
785,Data and code from - Identification of a key t...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251211045409/http...,"[Agriculture, Science & Research]",2026-07-29,/datasets/data-and-code-from---identification-...,"[{'id': 3941, 'status': 'Finished', 'download_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4635,Unpublished Board Decisions,National Labor Relations Board,National Labor Relations Board,<NA>,https://www.nlrb.gov/cases-decisions/decisions...,<NA>,nlrb.gov,False,<NA>,[Labor & Employment],2025-05-14,/datasets/unpublished-board-decisions/,"[{'id': 985, 'status': 'Finished', 'download_d..."
1390,Election Reports - FY 2014,National Labor Relations Board,National Labor Relations Board,<NA>,https://www.nlrb.gov/reports/agency-performanc...,<NA>,nlrb.gov,False,<NA>,[Labor & Employment],2025-04-01,/datasets/election-reports---fy-2014/,"[{'id': 665, 'status': 'Finished', 'download_d..."
1399,Election Reports - FY 2023,National Labor Relations Board,National Labor Relations Board,<NA>,https://www.nlrb.gov/reports/agency-performanc...,<NA>,nlrb.gov,False,<NA>,[Labor & Employment],2025-04-01,/datasets/election-reports---fy-2023/,"[{'id': 656, 'status': 'Finished', 'download_d..."
557,Climate Change Maps,U.S. Geological Survey,Department of the Interior,<NA>,https://www.sciencebase.gov/catalog/item/586d8...,<NA>,sciencebase.gov,False,<NA>,"[Climate & Environment, Infrastructure, Scienc...",2026-01-27,/datasets/climate-change-maps/,"[{'id': 2968, 'status': 'Finished', 'download_..."


## Statuses


In [4]:
from helpers import get_statuses, render_links

statuses = await get_statuses(client, sample["data_source"])
sample["status"] = statuses

render_links(sample[["data_source", "status"]].head())

,data_source,status
28,https://agdatacommons.nal.usda.gov/articles/dataset/2019_Farm_to_School_Census/25212905,🟢 202
735,https://agdatacommons.nal.usda.gov/articles/dataset/CSR2_Study_for_Greenhouse_gas_Reduction_through_Agricultural_Carbon_Enhancement_network_in_Watkinsville_Georgia/24665292,🟢 202
527,https://agdatacommons.nal.usda.gov/articles/dataset/Chelonus_insularis_Official_Gene_Set_OGSv1_0/24854583,🟢 202
647,https://agdatacommons.nal.usda.gov/articles/dataset/Comparison_of_four_extractants_used_in_soil_phosphorus_and_potassium_testing_for_two_soils_in_a_corn-wheat-soybean_rotation_in_Tennessee_receiving_various_amounts_of_P_and_K_fertilizer/24665652,🟢 202
785,https://agdatacommons.nal.usda.gov/articles/dataset/Data_and_code_from_Identification_of_a_key_target_for_elimination_of_nitrous_oxide_a_major_greenhouse_gas/24668340,🟢 202


### Source missing


In [5]:
is_error = sample["status"].str.startswith("🔴")
# assume this is a local issue
too_many_requests = sample["status"] == "🔴 429"

sample[is_error & ~too_many_requests][["data_source", "status"]]

,data_source,status
200,https://data.cdc.gov/Environmental-Health-Toxi...,🔴 404
759,https://data.cdc.gov/Environmental-Health-Toxi...,🔴 404
764,https://data.cdc.gov/Environmental-Health-Toxi...,🔴 404
461,https://data.cdc.gov/dataset/CDC-Library-Subsc...,🔴 404
4162,https://data.cdc.gov/dataset/Science-Clips/bii...,🔴 404
819,https://data.usgs.gov/datacatalog/data/USGS662...,🔴 405
3632,https://github.com/NREL-Sienna-Sienna,🔴 404
400,https://www.ahrq.gov/cahps/surveys-guidance/ic...,🔴 403
4242,https://www.ahrq.gov/sdoh/data-analytics.html,🔴 403
4270,https://www.ahrq.gov/sops/databases/medical-of...,🔴 403
